# 1_MNIST_CNN: CNN + MLP cho MNIST (chuyển từ Keras sang PyTorch)

3 khối Conv(64/128/256, 3×3, padding same) + MaxPool + Dropout 0.2 → Flatten → FC 512 → FC 512 → 10. SGD lr = 0.01, batch 128, 30 epoch. Sau khi train: trực quan hoá bằng PCA ảnh gốc, đặc trưng fc2 và đặc trưng pool3 (Global Average Pooling).

In [ ]:
!pip -q install torchinfo

In [ ]:
import os, time, random
import numpy as np
import torch, torch.nn as nn
import matplotlib.pyplot as plt
from google.colab import drive

drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/BTL03'
os.makedirs(BASE + '/P1', exist_ok=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(torch.__version__, device)

from torchvision import datasets

# (1) Dữ liệu

MNIST được nạp một lần lên GPU dưới dạng tensor (60000 × 1 × 28 × 28 float32 chỉ khoảng 190 MB) nên mỗi epoch rất nhanh. Keras dùng ảnh dạng (28, 28, 1), PyTorch dùng (1, 28, 28).

In [ ]:
tr = datasets.MNIST('./data', train=True,  download=True)
te = datasets.MNIST('./data', train=False, download=True)
x_train = tr.data.float().div(255).unsqueeze(1).to(device)   # (60000, 1, 28, 28)
y_train = tr.targets.to(device)
x_test  = te.data.float().div(255).unsqueeze(1).to(device)   # (10000, 1, 28, 28)
y_test  = te.targets.to(device)
print(x_train.shape, y_train.shape, x_test.shape, y_test.shape)

# (2) Mô hình

Thứ tự mỗi khối giống Keras: Conv → ReLU → MaxPool → Dropout. Sau 3 lần MaxPool, ảnh 28 → 14 → 7 → 3 nên Flatten ra 256 × 3 × 3 = 2304. `forward` trả thêm đặc trưng fc2 và đặc trưng pool3 (sau Global Average Pooling) để vẽ PCA.

In [ ]:
class MnistCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 64, 3, padding=1)
        self.conv2 = nn.Conv2d(64, 128, 3, padding=1)
        self.conv3 = nn.Conv2d(128, 256, 3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.drop = nn.Dropout(0.2)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(256 * 3 * 3, 512)
        self.fc2 = nn.Linear(512, 512)
        self.out = nn.Linear(512, num_classes)

    def forward(self, x, return_feats=False):
        x = self.drop(self.pool(torch.relu(self.conv1(x))))
        x = self.drop(self.pool(torch.relu(self.conv2(x))))
        p3 = self.pool(torch.relu(self.conv3(x)))        # = lớp pool3 của Keras
        x = self.flatten(self.drop(p3))
        x = self.drop(torch.relu(self.fc1(x)))
        f2 = torch.relu(self.fc2(x))                     # = lớp fc2 của Keras
        logits = self.out(self.drop(f2))                 # không có softmax
        if return_feats:
            return logits, f2, p3.mean(dim=(2, 3))       # GlobalAveragePooling2D
        return logits

model = MnistCNN().to(device)

from torchinfo import summary
summary(model, input_size=(1, 1, 28, 28))                # Total params phải là 1,817,610

# (3) Huấn luyện

In [ ]:
loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.SGD(model.parameters(), lr=0.01)

def iterate(x, y, bs, shuffle):
    idx = torch.randperm(x.size(0), device=x.device) if shuffle else torch.arange(x.size(0), device=x.device)
    for i in range(0, x.size(0), bs):
        j = idx[i:i + bs]
        yield x[j], y[j]

def run_epoch(model, x, y, opt=None, bs=128):
    train = opt is not None
    model.train(train)
    total, correct = 0.0, 0
    with torch.set_grad_enabled(train):
        for xb, yb in iterate(x, y, bs, shuffle=train):
            out = model(xb)
            loss = loss_fn(out, yb)
            if train:
                opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item() * xb.size(0)
            correct += (out.argmax(1) == yb).sum().item()
    n = x.size(0)
    return total / n, correct / n

In [ ]:
hist = {"loss": [], "acc": [], "val_loss": [], "val_acc": []}
t0 = time.time()
for ep in range(1, 31):
    l, a = run_epoch(model, x_train, y_train, opt)
    vl, va = run_epoch(model, x_test, y_test)
    for k, v in zip(hist, (l, a, vl, va)):
        hist[k].append(v)
    print(f"epoch {ep:2d} | loss {l:.4f} acc {a:.4f} | val_loss {vl:.4f} val_acc {va:.4f}")
print('> training time is %.4f minutes' % ((time.time() - t0) / 60))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(hist["loss"], label="train"); ax[0].plot(hist["val_loss"], label="val")
ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(hist["acc"], label="train"); ax[1].plot(hist["val_acc"], label="val")
ax[1].set_title("Accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.tight_layout()
plt.savefig(BASE + '/P1/mnist_cnn_curves.png', dpi=150)
plt.show()

# (4) Đánh giá mô hình đã huấn luyện

In [ ]:
test_loss, test_acc = run_epoch(model, x_test, y_test)
print('Test loss:', test_loss)
print('Test accuracy:', test_acc)

## PCA ảnh gốc (784 chiều → 2 chiều)

In [ ]:
import seaborn as sns
from sklearn.decomposition import PCA

data_lbl = y_train.cpu().numpy()
data_org = x_train.reshape(x_train.shape[0], -1).cpu().numpy()
print(data_org.shape, data_lbl.shape)

def pca_plot(feat, labels, title, fname):
    z = PCA(n_components=2).fit_transform(feat)
    print(z.shape)
    plt.figure(figsize=(8, 8))
    sns.scatterplot(x=z[:, 0], y=z[:, 1], hue=labels, palette='viridis', s=6, linewidth=0, legend='full')
    plt.xlabel('PC1'); plt.ylabel('PC2'); plt.title(title, fontsize=15)
    plt.savefig(BASE + '/P1/' + fname, dpi=150)
    plt.show()
    return z

_ = pca_plot(data_org, data_lbl, 'PCA of raw MNIST images', 'mnist_pca_raw.png')

## Dự đoán một ảnh test ngẫu nhiên

In [ ]:
model.eval()
i = np.random.randint(x_test.shape[0])
with torch.no_grad():
    pred_label = model(x_test[i:i + 1]).argmax(1).item()
plt.imshow(x_test[i, 0].cpu(), cmap='gray')
plt.title('true label = %d, predicted label = %d' % (y_test[i].item(), pred_label))
plt.axis('off'); plt.show()

## PCA đặc trưng fc2 và đặc trưng pool3 (Global Average Pooling)

Bản gốc dùng `Model(inputs, outputs=fc2)` và `GlobalAveragePooling2D` trên `pool3`; ở đây lấy trực tiếp từ `forward(return_feats=True)`. Cell `model.get_layer()` ở bản gốc gọi thiếu tham số nên không có tương đương.

In [ ]:
@torch.no_grad()
def extract(model, x, bs=1000):
    model.eval()
    f2s, gaps = [], []
    for i in range(0, x.size(0), bs):
        _, f2, gap = model(x[i:i + bs], return_feats=True)
        f2s.append(f2.cpu()); gaps.append(gap.cpu())
    return torch.cat(f2s).numpy(), torch.cat(gaps).numpy()

y_feat, y2_feat = extract(model, x_train)
print(y_feat.shape, y2_feat.shape)     # (60000, 512) (60000, 256)
_ = pca_plot(y_feat,  data_lbl, 'PCA of fc2 features', 'mnist_pca_fc2.png')
_ = pca_plot(y2_feat, data_lbl, 'PCA of pool3 features (GAP)', 'mnist_pca_pool3.png')

In [ ]:
torch.save(model.state_dict(), BASE + '/P1/mnist_cnn.pt')
print("Đã lưu")

# Điểm khác biệt Keras → PyTorch trong notebook này

- Ảnh NHWC (28, 28, 1) → NCHW (1, 28, 28); `padding='same'` → `padding=1`.
- `Conv2D/MaxPooling2D/Dropout/Dense` → `nn.Conv2d/nn.MaxPool2d/nn.Dropout/nn.Linear`; phải khai báo `in_channels` và `in_features` (2304 sau Flatten).
- `categorical_crossentropy` + softmax + `to_categorical` → `CrossEntropyLoss` trên logits với nhãn số.
- `fit/evaluate` → vòng lặp viết tay; `model.train()/eval()` bật/tắt Dropout.
- `Model(inputs, outputs=layer)` và `get_layer()` để lấy đặc trưng → `forward(return_feats=True)`.
- `GlobalAveragePooling2D` → `tensor.mean(dim=(2, 3))`.